In [1]:
import geopandas as gpd
import gcsfs
import pandas as pd

from google.cloud import bigquery

from shared_vars import RAW_GCS, INTERMED_GCS, AGENCY_TO_GTFS_NAME_DICT, RAW_DATA_YAML
from ridership_utils import bq_utils, utils

In [2]:
import download_gtfs
one_operator = "bart"

In [3]:
import clean_bart

bart_stops = download_gtfs.import_stops_for_operator(one_operator)

# Only need a subset of columns
keep_ridership_cols = [
    "Date", "day_type", 
    "Station", "stop_name", 
    "avg_boardings", "avg_alightings",
    "start_date", "end_date", "schedule_name"
]
bart_ridership = download_gtfs.prep_ridership_for_gtfs_join(one_operator).pipe(
    clean_bart.rename_operator_columns
)[keep_ridership_cols]

In [4]:
bart_ridership.shape

(18250, 9)

In [6]:
bart_ridership.head(2)

,Date,day_type,Station,stop_name,avg_boardings,avg_alightings,start_date,end_date,schedule_name
0,2024-10-01,Weekday,12,12th Street / Oakland City Center,5946,5918,2024-10-01,2024-10-01,Bay Area 511 BART Schedule
1,2024-10-01,Weekday,16,16th Street Mission,6259,6015,2024-10-01,2024-10-01,Bay Area 511 BART Schedule


In [8]:
bart_stops.head(2)

,key,feed_key,stop_id,stop_name,stop_code,geometry,schedule_name,service_date_start,service_date_end
0,5e105587872daca628e4b250996a1c18,cb97d26cb94b43f16a843999d823d63a,COLS_2,Enter/Exit : Station entrance (North),COLS_2,POINT (-122.19678 37.754),Bay Area 511 BART Schedule,2024-11-01,2024-11-04
1,4259a11cbc18254376c7784cda6a0044,cb97d26cb94b43f16a843999d823d63a,19TH_2,Enter/Exit : Broadway @ 17th Street (West),19TH_2,POINT (-122.26979 37.80728),Bay Area 511 BART Schedule,2024-11-01,2024-11-04


In [9]:
bart_stops2 = download_gtfs.dedupe_stops(bart_stops)

In [10]:
pd.merge(
    bart_ridership,
    bart_stops,
    on = ["schedule_name", "stop_name"],
    how = "left",
    indicator=True
)._merge.value_counts()

_merge
both          895345
left_only       2555
right_only         0
Name: count, dtype: int64

In [11]:
bart_gdf = pd.merge(
    bart_ridership,
    bart_stops2,
    on = ["schedule_name", "stop_name"],
    how = "left",
    indicator=True
)

bart_gdf._merge.value_counts()

_merge
both          78840
left_only      2555
right_only        0
Name: count, dtype: int64

In [ ]:
bart_ridership.stop_name.nunique()

In [ ]:
bart_gdf.stop_name.nunique()

In [12]:
# as is: stops have same location regardless of timeseries
# as is: 
bart_gdf[bart_gdf.stop_name=="MacArthur"]

,Date,day_type,Station,stop_name,avg_boardings,avg_alightings,start_date,end_date,schedule_name,stop_id,key,geometry,_merge
112,2024-10-01,Weekday,MA,MacArthur,4227,4132,2024-10-01,2024-10-01,Bay Area 511 BART Schedule,900301,4d407f64fadb3a5fa76ed53c04ff1daf,POINT (-122.26698 37.82879),both
113,2024-10-01,Weekday,MA,MacArthur,4227,4132,2024-10-01,2024-10-01,Bay Area 511 BART Schedule,900302,b01f099e10d872889d96094869556c0e,POINT (-122.26726 37.82883),both
114,2024-10-01,Weekday,MA,MacArthur,4227,4132,2024-10-01,2024-10-01,Bay Area 511 BART Schedule,900303,9a2b4a346e60fb84f8a7d5f8d2392a26,POINT (-122.26708 37.8288),both
115,2024-10-01,Weekday,MA,MacArthur,4227,4132,2024-10-01,2024-10-01,Bay Area 511 BART Schedule,900304,46bd090f7a95f7286adf3a290aee5ca7,POINT (-122.26716 37.82881),both
116,2024-10-01,Weekday,MA,MacArthur,4227,4132,2024-10-01,2024-10-01,Bay Area 511 BART Schedule,900309,0df91ae2e03bbffc69c6356569b7f563,POINT (-122.2671 37.8288),both
...,...,...,...,...,...,...,...,...,...,...,...,...,...
81286,2025-09-30,Weekday,MA,MacArthur,4754,4615,2025-09-30,2025-09-30,Bay Area 511 BART Schedule,900303,9a2b4a346e60fb84f8a7d5f8d2392a26,POINT (-122.26708 37.8288),both
81287,2025-09-30,Weekday,MA,MacArthur,4754,4615,2025-09-30,2025-09-30,Bay Area 511 BART Schedule,900304,46bd090f7a95f7286adf3a290aee5ca7,POINT (-122.26716 37.82881),both
81288,2025-09-30,Weekday,MA,MacArthur,4754,4615,2025-09-30,2025-09-30,Bay Area 511 BART Schedule,900309,0df91ae2e03bbffc69c6356569b7f563,POINT (-122.2671 37.8288),both
81289,2025-09-30,Weekday,MA,MacArthur,4754,4615,2025-09-30,2025-09-30,Bay Area 511 BART Schedule,MCAR,459750286cc9cad47909bebf96df9feb,POINT (-122.26716 37.82881),both
